# Home Assignment 4 — CS5720 Neural Network and Deep Learning
**Student Name:** Palash Palit  
Fall 2026, University of Central Missouri

Run all cells (**Runtime → Run all**). The written answers are in the README.

## Setup: install the libraries

In [ ]:
!pip install -q nltk spacy
!python -m spacy download en_core_web_sm

## Q1: NLP Preprocessing Pipeline

In [ ]:
"""
Home Assignment 4 - Part II, Q1: NLP Preprocessing Pipeline
CS5720 Neural Network and Deep Learning - Fall 2026
Student: Palash Palit

Steps: 1) tokenize  2) remove English stopwords  3) apply Porter stemming
"""
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download the NLTK data the pipeline needs, only if it is not installed yet
for path, resource in [("tokenizers/punkt", "punkt"), ("tokenizers/punkt_tab", "punkt_tab"),
                       ("corpora/stopwords", "stopwords")]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(resource, quiet=True)


def preprocess(sentence):
    """Tokenize, remove stopwords and stem a sentence; print the result of each step."""
    # Step 1: Tokenization - split the sentence into words and punctuation marks
    tokens = word_tokenize(sentence)
    print("1. Original Tokens:", tokens)

    # Step 2: Stopword removal - drop common words ("the", "in", "are", ...).
    # Comparison is lowercase so "And" and "and" are both removed.
    # Punctuation is also dropped here because it carries no meaning on its own.
    stop_words = set(stopwords.words("english"))
    filtered = [t for t in tokens if t.lower() not in stop_words and t.isalnum()]
    print("2. Tokens Without Stopwords:", filtered)

    # Step 3: Stemming - cut each word down to its root with the Porter stemmer
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(t) for t in filtered]
    print("3. Stemmed Words:", stemmed)
    return tokens, filtered, stemmed


if __name__ == "__main__":
    sentence = "NLP techniques are used in virtual assistants like Alexa and Siri."
    print("Input sentence:", sentence)
    print()
    preprocess(sentence)


## Q2: Named Entity Recognition with spaCy

In [ ]:
"""
Home Assignment 4 - Part II, Q2: Named Entity Recognition with spaCy
CS5720 Neural Network and Deep Learning - Fall 2026
Student: Palash Palit

Setup (one time):
    pip install spacy
    python -m spacy download en_core_web_sm
"""
import spacy

# Load spaCy's small English pipeline, which includes a trained NER component
nlp = spacy.load("en_core_web_sm")

# Input sentence exactly as given in the assignment (including the leading space)
sentence = " LeBron James as the NBA player of the United States in Laker and won the NBA Final MVP in 2020."

# Run the pipeline: tokenization, tagging, parsing and named entity recognition
doc = nlp(sentence)

print("Sentence:", repr(sentence))
print()
print(f"{'Entity text':<22}{'Label':<10}{'Start':>6}{'End':>6}   Label meaning")
print("-" * 70)
for ent in doc.ents:
    # ent.start_char / ent.end_char are character positions in the original string
    # (end is exclusive, so sentence[start:end] gives back the entity text)
    print(f"{ent.text:<22}{ent.label_:<10}{ent.start_char:>6}{ent.end_char:>6}   {spacy.explain(ent.label_)}")
    assert sentence[ent.start_char:ent.end_char] == ent.text   # check the positions are correct

print()
print("Total entities found:", len(doc.ents))


## Q3: Scaled Dot-Product Attention

In [ ]:
"""
Home Assignment 4 - Part II, Q3: Scaled Dot-Product Attention
CS5720 Neural Network and Deep Learning - Fall 2026
Student: Palash Palit

Attention(Q, K, V) = softmax(Q K^T / sqrt(d)) V
"""
import numpy as np

np.set_printoptions(precision=4, suppress=True)


def softmax(x):
    """Row-wise softmax. Subtracting the row max first avoids overflow and does not change the result."""
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """Return (attention_weights, output) for query, key and value matrices."""
    d = K.shape[-1]                      # key dimension
    scores = Q @ K.T                     # 1) dot product of Q and K transpose
    scaled = scores / np.sqrt(d)         # 2) scale by sqrt(d)
    weights = softmax(scaled)            # 3) softmax -> attention weights (each row sums to 1)
    output = weights @ V                 # 4) weighted sum of the value vectors
    return scores, scaled, weights, output


if __name__ == "__main__":
    Q = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
    K = np.array([[1, 0, 1, 0], [0, 1, 0, 1]])
    V = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])

    scores, scaled, weights, output = scaled_dot_product_attention(Q, K, V)
    print("Q K^T (raw scores):\n", scores)
    print(f"\nScaled scores (divided by sqrt(d) = sqrt({K.shape[-1]}) = {np.sqrt(K.shape[-1]):.0f}):\n", scaled)
    print("\n1. Attention weights (after softmax):\n", weights)
    print("   Row sums:", weights.sum(axis=1))
    print("\n2. Final output (weights x V):\n", output)
